<a href="https://colab.research.google.com/github/SohailAkhtar466/-Urdu-AI-Voice-Over-Generator/blob/main/Urdu_AI_Voice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Step 1: Libraries Install Karna**

Pehle cell ma hum zaroori libraries install karenge. edge-tts audio ke liye aur streamlit website ke interface ke liye.

In [1]:
import os
!pip install edge-tts streamlit pyngrok -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 37.7 MB/s eta 0:00:00


**Step 2: app.py File Banana (Magic Command)**

In [2]:
%%writefile app.py
import streamlit as st
import edge_tts
import asyncio
import os
import base64

# --- 1. Page Configuration (Must be first) ---
st.set_page_config(
    page_title="SpeakSay AI Clone",
    page_icon="🎙️",
    layout="wide",  # "Wide" mode for full-screen SaaS look
    initial_sidebar_state="expanded"
)

# --- 2. Custom CSS for "SpeakSay" Look ---
# Ye CSS pure interface ko Modern/Dark SaaS look degi
st.markdown("""
<style>
    /* Main Background */
    .stApp {
        background-color: #0E1117; /* Dark Background */
        color: #FFFFFF;
    }

    /* Text Area Styling */
    .stTextArea textarea {
        background-color: #1E1E1E !important;
        color: #FFFFFF !important;
        border: 1px solid #333333;
        border-radius: 10px;
        font-size: 18px;
        font-family: 'Arial', sans-serif;
    }

    /* Buttons Styling (Purple Gradient like SpeakSay) */
    .stButton>button {
        background: linear-gradient(90deg, #7C3AED 0%, #5B21B6 100%);
        color: white;
        border: none;
        padding: 12px 24px;
        border-radius: 8px;
        font-weight: bold;
        width: 100%;
        transition: 0.3s;
    }
    .stButton>button:hover {
        opacity: 0.9;
        transform: scale(1.02);
    }

    /* Sidebar Styling */
    section[data-testid="stSidebar"] {
        background-color: #161B22;
        border-right: 1px solid #30363D;
    }

    /* Headers */
    h1, h2, h3 {
        color: #F0F6FC !important;
        font-family: 'Inter', sans-serif;
    }

    /* Custom Card for Voice Selector */
    .voice-card {
        background-color: #1E1E1E;
        padding: 15px;
        border-radius: 10px;
        border: 1px solid #333333;
        margin-bottom: 20px;
    }
</style>
""", unsafe_allow_html=True)

# --- 3. Sidebar Navigation (Left Panel) ---
with st.sidebar:
    st.image("https://cdn-icons-png.flaticon.com/512/4712/4712009.png", width=50) # Placeholder Logo
    st.markdown("### **SpeakSay Studio**")
    st.markdown("---")

    menu = st.radio("Navigation", ["🎙️ Text to Speech", "📂 History (Saved)", "👤 My Account"])

    st.markdown("---")
    st.markdown("#### **Voice Settings**")

    # Voice Selector imitating a "Pro" dropdown
    gender = st.selectbox("Select Voice Actor", ["👨 Male (Asad - Urdu)", "👩 Female (Uzma - Urdu)"])

    # Advanced Sliders
    with st.expander("🎚️ Advanced Controls", expanded=True):
        rate = st.slider("Speed", -50, 50, 0, format="%d%%")
        pitch = st.slider("Pitch", -50, 50, 0, format="%dHz")

# --- 4. Main Content Area (Right Panel) ---

if menu == "🎙️ Text to Speech":
    # Header Section
    col1, col2 = st.columns([3, 1])
    with col1:
        st.title("Create New Voiceover")
        st.caption("Transform your text into lifelike Urdu speech instantly.")

    # Logic for Voice ID
    VOICE_ID = "ur-PK-AsadNeural" if "Male" in gender else "ur-PK-UzmaNeural"
    rate_str = f"{rate:+d}%"
    pitch_str = f"{pitch:+d}Hz"

    # --- The Editor Interface (Split Layout) ---
    c1, c2 = st.columns([2, 1])

    with c1:
        st.markdown("#### **Script Editor**")
        text_input = st.text_area(
            "Type or paste your script here...",
            height=350,
            label_visibility="collapsed",
            placeholder="Assalam-o-Alaikum, aaj hum baat karenge YouTube Automation ke baare mein..."
        )

        # Character Counter
        st.caption(f"Characters: {len(text_input)} / Unlimited")

    with c2:
        st.markdown("#### **Preview & Generate**")
        st.markdown("""
        <div class="voice-card">
            <p style="margin:0; color:#888; font-size:12px;">SELECTED VOICE</p>
            <h3 style="margin:0;">Example Urdu</h3>
            <p style="color:green; font-size:12px;">● High Quality Neural</p>
        </div>
        """, unsafe_allow_html=True)

        generate_btn = st.button("✨ Generate Audio", type="primary")

        # Placeholder for Audio
        audio_placeholder = st.empty()

    # --- 5. Backend Logic ---
    async def generate_audio_logic(text, voice, rate, pitch):
        output_file = "output.mp3"
        communicate = edge_tts.Communicate(text, voice, rate=rate, pitch=pitch)
        await communicate.save(output_file)
        return output_file

    if generate_btn:
        if not text_input.strip():
            st.error("⚠️ Please enter some text to generate audio.")
        else:
            with st.spinner("Processing AI Voice..."):
                try:
                    # Run Async Function
                    loop = asyncio.new_event_loop()
                    asyncio.set_event_loop(loop)
                    loop.run_until_complete(generate_audio_logic(text_input, VOICE_ID, rate_str, pitch_str))

                    # Display Audio
                    with c2:
                        st.success("Rendering Complete!")
                        audio_file = open("output.mp3", 'rb')
                        audio_bytes = audio_file.read()

                        st.audio(audio_bytes, format='audio/mp3')

                        # Download Button
                        st.download_button(
                            label="📥 Download MP3",
                            data=audio_bytes,
                            file_name="speaksay_clone_urdu.mp3",
                            mime="audio/mp3"
                        )
                except Exception as e:
                    st.error(f"Error: {e}")

elif menu == "📂 History (Saved)":
    st.title("Your Library")
    st.info("Apki pichli generations yahan show hongi (Database connect karne ke baad).")

elif menu == "👤 My Account":
    st.title("User Profile")
    st.write("Name: Python Developer")
    st.write("Plan: **Free Unlimited (Engineer Edition)**")

Writing app.py


**Step 3: Website ko Live Karna (Tunneling)**

In [3]:
# Yahan apna Ngrok Auth Token paste karein.
# Example: os.environ["NGROK_AUTH_TOKEN"] = "2mF1E_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"
os.environ["NGROK_AUTH_TOKEN"] = "2xDqTwuxhjGv2oQsh91LwG9j2Jl_7Jos9L4fVuBjTsSxqNTBz" # Replace "YOUR_AUTH_TOKEN" with your actual token

import asyncio
from pyngrok import ngrok, conf
import nest_asyncio # Import nest_asyncio
import os
import time
import subprocess

# Apply nest_asyncio to allow asyncio.run in an already running loop in Colab
nest_asyncio.apply()

app_file = "app.py"

# Check if app.py exists before trying to run Streamlit
if not os.path.exists(app_file):
    print(f"Error: '{app_file}' not found. Please ensure 'Step 2: app.py File Banana (Magic Command)' cell is executed successfully.")
else:
    print(f"Running Streamlit app '{app_file}' in the background...")
    # Use subprocess.Popen for more controlled execution and error handling of Streamlit
    # We redirect stdout/stderr to pipes so we can capture them if needed later
    streamlit_process = subprocess.Popen(
        ["streamlit", "run", app_file, "--server.port", "8501", "--server.enableCORS", "false", "--server.enableXsrfProtection", "false"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )
    # Give streamlit a moment to start up and bind to its port
    time.sleep(5)
    print("Streamlit process started. Attempting to create ngrok tunnel...")

    async def setup_ngrok():
        try:
            # Ensure auth token is set
            if "NGROK_AUTH_TOKEN" not in os.environ or os.environ["NGROK_AUTH_TOKEN"] == "YOUR_AUTH_TOKEN":
                print("Error: NGROK_AUTH_TOKEN not set. Please get your token from https://dashboard.ngrok.com/get-started/your-authtoken and replace 'YOUR_AUTH_TOKEN' in the code.")
                # If ngrok auth is missing, terminate the streamlit process as we can't tunnel
                streamlit_process.terminate()
                return

            ngrok_auth_token = os.environ["NGROK_AUTH_TOKEN"]
            conf.get_default().auth_token = ngrok_auth_token

            # Kill any existing ngrok tunnels to avoid conflicts
            ngrok.kill()

            # Open a ngrok tunnel to the Streamlit port (8501)
            public_url = ngrok.connect(8501, bind_tls=True)
            print(f"\nStreamlit App is Live on: {public_url}")
            print(f"Ngrok Web Interface: http://127.0.0.1:4040\n")

        except Exception as e:
            print(f"\nNgrok Tunnel creation failed: {e}")
            # If ngrok setup fails, capture and print any output from Streamlit for debugging
            stdout, stderr = streamlit_process.communicate()
            if stdout:
                print("\n--- Streamlit STDOUT ---")
                print(stdout.decode())
            if stderr:
                print("\n--- Streamlit STDERR ---")
                print(stderr.decode())
            streamlit_process.terminate() # Ensure streamlit process is terminated if ngrok fails

    # Run the async function
    asyncio.run(setup_ngrok())


Running Streamlit app 'app.py' in the background...
Streamlit process started. Attempting to create ngrok tunnel...

Streamlit App is Live on: NgrokTunnel: "https://711b-34-86-176-193.ngrok-free.app" -> "http://localhost:8501"
Ngrok Web Interface: http://127.0.0.1:4040

